# Chapter 16 — SQLite Databases

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 16](https://automatetheboringstuff.com/3e/chapter16.html)

**Setup note:** Uses `sqlite3` from the standard library — fully runnable, no install needed.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 16.1 Connecting and creating a table

SQLite stores an entire database in a single file. `sqlite3.connect(path)` opens (or creates) it; `conn.cursor()` gives you an object to run SQL through `.execute()`. `CREATE TABLE` defines the columns and types.


In [1]:
import sqlite3

conn = sqlite3.connect('demo.db')
cur = conn.cursor()
cur.execute('CREATE TABLE IF NOT EXISTS cats (id INTEGER PRIMARY KEY, name TEXT, age INTEGER)')
conn.commit()
print('table ready')


table ready


## 16.2 INSERT and parameterized queries

Always pass values as a separate tuple with `?` placeholders instead of building SQL strings with f-strings/`+` — this prevents SQL injection and correctly handles quoting for you.


In [2]:
import sqlite3
conn = sqlite3.connect('demo.db')
cur = conn.cursor()

cur.execute('INSERT INTO cats (name, age) VALUES (?, ?)', ('Zophie', 8))
cur.execute('INSERT INTO cats (name, age) VALUES (?, ?)', ('Pooka', 5))
conn.commit()
print('rows inserted')


rows inserted


## 16.3 SELECT, UPDATE, and DELETE

`SELECT` retrieves rows (`cur.fetchall()` / `cur.fetchone()`); `UPDATE` modifies existing rows matching a `WHERE` clause; `DELETE` removes them. Forgetting `WHERE` on an UPDATE/DELETE affects every row — always double-check it.


In [3]:
import sqlite3
conn = sqlite3.connect('demo.db')
cur = conn.cursor()

cur.execute('SELECT * FROM cats')
print(cur.fetchall())

cur.execute('UPDATE cats SET age = ? WHERE name = ?', (9, 'Zophie'))
conn.commit()
cur.execute('SELECT * FROM cats WHERE name = ?', ('Zophie',))
print(cur.fetchone())

cur.execute('DELETE FROM cats WHERE name = ?', ('Pooka',))
conn.commit()
cur.execute('SELECT * FROM cats')
print(cur.fetchall())

conn.close()
import os
os.remove('demo.db')  # clean up


[(1, 'Zophie', 8), (2, 'Pooka', 5)]
(1, 'Zophie', 9)
[(1, 'Zophie', 9)]


## 16.4 Why parameterized queries matter

Building SQL with string concatenation lets user input change the *structure* of your query (SQL injection) — a classic, still-common vulnerability. The `?` placeholder approach passes values as data only, never as executable SQL, which closes that door entirely.


In [4]:
unsafe_input = "'; DROP TABLE cats; --"
# NEVER do this:
# cur.execute(f"SELECT * FROM cats WHERE name = '{unsafe_input}'")   # vulnerable!
# Always do this instead:
# cur.execute('SELECT * FROM cats WHERE name = ?', (unsafe_input,))  # safe: treated as plain text
print('Parameterized queries treat input as data, never as code.')


Parameterized queries treat input as data, never as code.


## 16.5 Foreign keys and INNER JOIN across tables

Real databases split data across multiple related tables instead of repeating information in one giant table — a `books` table might reference an `authors` table by an `author_id` **foreign key** instead of spelling out the author's name on every row. SQLite doesn't enforce foreign keys by default (`PRAGMA foreign_keys = ON` turns that on per-connection). To combine matching rows from two tables in a single query, use `INNER JOIN table2 ON table1.some_id = table2.id` — it returns only rows that have a match in both tables, which is the standard way to "look up" the author's actual name instead of just seeing their id number.

In [5]:
import sqlite3

conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')
cur = conn.cursor()

cur.execute('CREATE TABLE authors (id INTEGER PRIMARY KEY, name TEXT)')
cur.execute('CREATE TABLE books (id INTEGER PRIMARY KEY, title TEXT, author_id INTEGER, '
            'FOREIGN KEY (author_id) REFERENCES authors(id))')

cur.executemany('INSERT INTO authors (id, name) VALUES (?, ?)',
                [(1, 'Al Sweigart'), (2, 'Eric Matthes')])
cur.executemany('INSERT INTO books (title, author_id) VALUES (?, ?)',
                [('Automate the Boring Stuff', 1), ('Python Crash Course', 2), ('Beyond the Basic Stuff', 1)])
conn.commit()

# Without a JOIN, you only see the raw foreign key number -- not useful on its own
for row in cur.execute('SELECT title, author_id FROM books'):
    print('raw:', row)

print('---')

# INNER JOIN resolves author_id into the actual author name
cur.execute('''
    SELECT books.title, authors.name
    FROM books
    INNER JOIN authors ON books.author_id = authors.id
''')
for row in cur.fetchall():
    print('joined:', row)

conn.close()


raw: ('Automate the Boring Stuff', 1)
raw: ('Python Crash Course', 2)
raw: ('Beyond the Basic Stuff', 1)
---
joined: ('Automate the Boring Stuff', 'Al Sweigart')
joined: ('Python Crash Course', 'Eric Matthes')
joined: ('Beyond the Basic Stuff', 'Al Sweigart')
